# Video Generation Models

**Module:** 18 — Video Generation

Sora, Veo, Runway, Stability, Movie Gen, open-source lines — integration patterns and bakeoffs.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Compare leading video model offerings at a product level
- Design a provider-abstracted video generation client
- Run a structured bakeoff with motion-aware rubrics
- Plan fallbacks when a provider rate-limits or filters


## Landscape snapshot

| Line | Positioning | Notes |
|------|-------------|-------|
| **OpenAI Sora** | Frontier txt/img→video | Strong sim prior narrative; API/product access varies |
| **Google Veo** | Frontier + Google cloud story | Ecosystem with Gemini tooling |
| **Runway** | Creator-tooling + models | Gen lines, editing-oriented UX |
| **Stability** | Open/commercial video stacks | Self-host potential depends on release |
| **Meta Movie Gen** | Research / ecosystem signals | Watch licensing & availability |
| **Open source** | CogVideo, Open-Sora-class, etc. | Max control; you own MLOps |

```mermaid
flowchart LR
  APP --> IFACE[VideoProvider interface]
  IFACE --> SORA[Sora-class API]
  IFACE --> VEO[Veo API]
  IFACE --> RW[Runway API]
  IFACE --> OS[Self-hosted open]
```


## Integration Pattern

### Definition
Treat video like async jobs: create → poll/webhook → download → safety → store → present. Normalize provider schemas behind one interface.

### Why it matters
Sync HTTP timeouts and heterogeneous payloads otherwise wreck your app layer.

### How it works
Map inputs (prompt, image, duration, aspect, seed) to provider fields; normalize outputs to `{job_id, status, url, captions, model}`.

### Intuition
Airline booking APIs — different carriers, one itinerary object.

### Pitfalls
- Assuming identical content filters across providers
- No idempotent job keys on retries

### When to use
Any multi-provider or enterprise integration.


In [ ]:
# Demo 1: provider protocol + mocks
from typing import Protocol

class VideoProvider(Protocol):
    name: str
    def create(self, prompt: str, **kw) -> dict: ...
    def poll(self, job_id: str) -> dict: ...

class MockSora:
    name = "sora"
    def create(self, prompt: str, **kw) -> dict:
        return {"id": "sora_1", "status": "queued", "prompt": prompt, "seconds": kw.get("seconds", 4)}
    def poll(self, job_id: str) -> dict:
        return {"id": job_id, "status": "completed", "url": "https://example.invalid/sora.mp4"}

class MockRunway:
    name = "runway"
    def create(self, prompt: str, **kw) -> dict:
        return {"task_id": "rw_1", "state": "PENDING"}
    def poll(self, job_id: str) -> dict:
        return {"task_id": job_id, "state": "SUCCEEDED", "output": ["https://example.invalid/rw.mp4"]}

def normalize(provider: str, payload: dict) -> dict:
    if provider == "sora":
        return {"job_id": payload.get("id"), "status": payload.get("status"), "url": payload.get("url")}
    return {
        "job_id": payload.get("task_id"),
        "status": "completed" if payload.get("state") == "SUCCEEDED" else payload.get("state", "").lower(),
        "url": (payload.get("output") or [None])[0],
    }

s = MockSora(); print(normalize("sora", s.poll(s.create("waves")["id"])))
r = MockRunway(); print(normalize("runway", r.poll(r.create("waves")["task_id"])))


In [ ]:
# Demo 2: bakeoff scorecard for video
W = {"adherence": 0.25, "motion": 0.25, "identity": 0.2, "flicker": 0.15, "cost": 0.15}

def vscore(row):
    # flicker/cost already oriented as "higher is better"
    return sum(row[k]*w for k,w in W.items())

board = {
    "sora_class": {"adherence": 0.9, "motion": 0.9, "identity": 0.85, "flicker": 0.8, "cost": 0.4},
    "runway_class": {"adherence": 0.8, "motion": 0.82, "identity": 0.8, "flicker": 0.75, "cost": 0.55},
    "open_selfhost": {"adherence": 0.7, "motion": 0.7, "identity": 0.75, "flicker": 0.65, "cost": 0.9},
}
print(sorted(((k, round(vscore(v),3)) for k,v in board.items()), key=lambda x: -x[1]))


### Model notes (preserve coverage)

**OpenAI Sora** — Flagship long-horizon narrative demos; integrate with `YOUR_OPENAI_API_KEY` when available; treat duration/resolution as first-class quotas.

**Google Veo** — Strong photoreal/motion story; evaluate GCP data regions and safety.

**Runway** — Iterative creator workflows; good fit when editing + gen colocate.

**Stability** — Watch open weights vs API; Control-style ecosystems may differ from image SD maturity.

**Meta Movie Gen** — Track research→product path; verify access.

**Open source** — Best for air-gap and custom controls; you own drift and safety.

### Selection tips
- Ads with brand stills → img2video-strong providers
- Tooling for editors → Runway-like UX or NLE plugins
- Research reproducibility → open weights
- Always keep a fallback provider + offline apology path


In [ ]:
# Demo 3: request shapes (placeholders)
GOOGLE_API_KEY = "YOUR_GOOGLE_API_KEY"
RUNWAY_API_KEY = "YOUR_RUNWAY_API_KEY"
veo_req = {"prompt": "timelapse of city sunset", "durationSeconds": 6, "aspectRatio": "16:9"}
runway_req = {"promptText": "timelapse of city sunset", "model": "gen3a_turbo", "duration": 5}
print(GOOGLE_API_KEY[:8], veo_req)
print(RUNWAY_API_KEY[:8], runway_req)


In [ ]:
# Demo 4: filter / fallback router
def route(prompt: str, primary_status: str) -> str:
    banned = ["real person deepfake", "violent gore"]
    if any(b in prompt.lower() for b in banned):
        return "block"
    if primary_status in {"rate_limited", "outage"}:
        return "fallback_provider"
    if primary_status == "safety_filtered":
        return "rewrite_prompt_or_human"
    return "primary"

for p, st in [("ocean waves", "ok"), ("ocean waves", "rate_limited"), ("real person deepfake", "ok")]:
    print(p, st, "->", route(p, st))


### Try it yourself — Models

1. Implement webhook completion handler with signature verification stub.
2. Add latency & $ columns to the bakeoff board.
3. Write legal checklist: training-use of uploads, commercial rights, likeness.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `poll` | Client repeatedly checks job status |
| `webhook` | Provider pushes completion event |
| `bakeoff` | Fixed-suite provider comparison |
| `fallback` | Secondary provider/path on failure |


### Workshop — Parameter journal — Video Models

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Video Models
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Video Models

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Video Models
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Video Models

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Video Models
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Video Models

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Video Models
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Video Models

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Video Models
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Video Models

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Video Models
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


### Workshop — Interface sketch — Video Models

Sketch provider-agnostic request/response dicts for this modality.


In [ ]:
# Workshop 7 — Video Models
req = {'prompt':'...','size':'...','seed':1}
resp = {'status':'ok','asset_uri':'...','model':'...','seed':1}
print(req); print(resp)


### Workshop — Self-check — Video Models

Run this checklist printer and fill it honestly before moving on.


In [ ]:
# Workshop 8 — Video Models
for i,x in enumerate(['restated objectives','ran demos','named pitfall','named metric'],1):
    print(f'{i}. [ ] {x}')


## Key Takeaways

- Normalize providers behind async job interfaces
- Score motion and identity, not marketing stills
- Plan filters, rate limits, and fallbacks explicitly
